# Sesión 7 · Actividad 7
# Prompt Challenge: diseña, experimenta y justifica

**Curso:** Introducción a la Inteligencia Artificial Generativa  
**Modalidad:** reto integrador en Google Colab

---

## Propósito

En esta actividad integrarás los conceptos experimentados durante la sesión:

- anatomía del prompt;
- Guidance Scale;
- pasos de inferencia;
- scheduler;
- semilla;
- negative prompt.

Tu objetivo será diseñar una generación visual bajo restricciones concretas y justificar cada decisión.

> **Esta actividad no busca solamente una imagen atractiva.**  
> Busca una imagen reproducible, documentada y técnicamente justificada.


Esta actividad funciona como cierre integrador.

El alumnado deberá demostrar que comprende que una generación no depende de un único parámetro, sino de un conjunto de decisiones.

### Principio central

> **Generar no es presionar un botón: es diseñar un experimento.**

Antes de iniciar, recuerda:

- cambiar una variable a la vez durante las pruebas;
- registrar todas las configuraciones;
- conservar la semilla;
- justificar las decisiones con evidencia.


# 0. Preparación del entorno

En Google Colab selecciona:

**Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

Después ejecuta las celdas en orden.

Este cuaderno es completamente independiente y no requiere ninguna actividad anterior.


In [ ]:
import torch

print("Versión de PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No se detectó una GPU. Selecciona: "
        "Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU."
    )


# 1. Configuración del guardado

Puedes guardar los resultados:

- temporalmente en Colab;
- permanentemente en Google Drive.

Para guardar en Drive cambia `USAR_DRIVE` a `True`.


In [ ]:
from pathlib import Path

USAR_DRIVE = False

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_RESULTADOS = Path(
        "/content/drive/MyDrive/IA_Generativa/Sesion7/Actividad7_PromptChallenge"
    )
else:
    CARPETA_RESULTADOS = Path("/content/Actividad7_PromptChallenge")

CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
print("Los resultados se guardarán en:", CARPETA_RESULTADOS)


# 2. Instalación de bibliotecas

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors


# 3. Carga del modelo

Usaremos Stable Diffusion 1.5.

El cuaderno permitirá elegir entre:

- DDIM;
- Euler;
- Euler ancestral;
- Heun;
- DPM-Solver++.


In [ ]:
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from diffusers import (
    StableDiffusionPipeline,
    DDIMScheduler,
    EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler,
    HeunDiscreteScheduler,
    DPMSolverMultistepScheduler
)

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
    safety_checker=None
)

pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

CONFIG_BASE = pipe.scheduler.config

SCHEDULERS = {
    "DDIM": DDIMScheduler.from_config(CONFIG_BASE),
    "Euler": EulerDiscreteScheduler.from_config(CONFIG_BASE),
    "Euler ancestral": EulerAncestralDiscreteScheduler.from_config(CONFIG_BASE),
    "Heun": HeunDiscreteScheduler.from_config(CONFIG_BASE),
    "DPM-Solver++": DPMSolverMultistepScheduler.from_config(
        CONFIG_BASE,
        algorithm_type="dpmsolver++"
    )
}

print("Modelo y schedulers cargados correctamente.")


# 4. El desafío

## Encargo creativo

Debes generar una imagen para la portada ficticia de una exposición titulada:

### **“Ciudades del futuro: convivencia entre naturaleza y tecnología”**

La imagen debe incluir:

- una ciudad futurista;
- vegetación integrada en los edificios;
- personas utilizando el espacio;
- iluminación cinematográfica;
- composición visual clara;
- ausencia de texto o logotipos visibles.

## Restricciones técnicas

- máximo 35 pasos;
- Guidance Scale entre 4 y 12;
- una semilla registrada;
- un scheduler justificado;
- negative prompt de máximo 10 conceptos;
- tamaño 512 × 512;
- máximo cuatro generaciones de prueba.


# 5. Plan previo

Antes de generar, completa:

### Objetivo visual
¿Qué debe comunicar la imagen?

### Sujeto principal
¿Qué elemento será el centro de atención?

### Contexto
¿Dónde ocurre la escena?

### Estilo
¿Qué apariencia tendrá?

### Composición
¿Cómo se organizarán los elementos?

### Restricciones
¿Qué defectos o elementos deben evitarse?

### Hipótesis
¿Qué combinación de parámetros crees que funcionará mejor?


# 6. Configuración de la primera propuesta

Modifica los siguientes valores.

> **Importante:** registra cada cambio. No modifiques todas las variables al mismo tiempo durante las iteraciones.


In [ ]:
# PROMPT POSITIVO
PROMPT = (
    "A sustainable futuristic city with vertical gardens integrated into "
    "modern buildings, people walking through public spaces, cinematic lighting, "
    "wide composition, realistic, highly detailed"
)

# NEGATIVE PROMPT: máximo 10 conceptos
NEGATIVE_PROMPT = (
    "text, logo, watermark, blurry, low quality, empty streets, "
    "deformed people, duplicate, oversaturated, cropped"
)

# PARÁMETROS
SEED = 2026
STEPS = 25
GUIDANCE = 7.5
SCHEDULER_NAME = "DPM-Solver++"

WIDTH = 512
HEIGHT = 512

# Validaciones básicas
if not 4 <= GUIDANCE <= 12:
    raise ValueError("Guidance Scale debe estar entre 4 y 12.")

if not 1 <= STEPS <= 35:
    raise ValueError("El número de pasos debe ser como máximo 35.")

conceptos_negativos = [
    x.strip() for x in NEGATIVE_PROMPT.split(",") if x.strip()
]

if len(conceptos_negativos) > 10:
    raise ValueError(
        f"El negative prompt contiene {len(conceptos_negativos)} conceptos. "
        "El máximo permitido es 10."
    )

if SCHEDULER_NAME not in SCHEDULERS:
    raise ValueError(
        f"Scheduler no válido. Opciones: {list(SCHEDULERS.keys())}"
    )

print("Configuración válida.")
print("Número de conceptos negativos:", len(conceptos_negativos))


# 7. Función de generación y registro

Cada prueba se guardará junto con sus parámetros.

Esto permitirá reproducirla posteriormente.


In [ ]:
historial = []

def generar_prueba(
    numero_prueba,
    prompt,
    negative_prompt,
    seed,
    steps,
    guidance,
    scheduler_name
):
    pipe.scheduler = SCHEDULERS[scheduler_name]
    generator = torch.Generator(device="cuda").manual_seed(seed)

    inicio = time.time()

    imagen = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        width=WIDTH,
        height=HEIGHT,
        generator=generator
    ).images[0]

    duracion = time.time() - inicio

    ruta_imagen = CARPETA_RESULTADOS / f"prueba_{numero_prueba}.png"
    imagen.save(ruta_imagen)

    registro = {
        "prueba": numero_prueba,
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "seed": seed,
        "steps": steps,
        "guidance": guidance,
        "scheduler": scheduler_name,
        "tiempo_segundos": round(duracion, 2),
        "archivo": str(ruta_imagen)
    }

    historial.append(registro)

    return imagen, registro


# 8. Prueba 1: línea base

Genera tu primera propuesta.


In [ ]:
imagen_1, registro_1 = generar_prueba(
    numero_prueba=1,
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    seed=SEED,
    steps=STEPS,
    guidance=GUIDANCE,
    scheduler_name=SCHEDULER_NAME
)

display(imagen_1)
registro_1


## Evaluación de la prueba 1

Asigna una calificación de 1 a 5.

| Criterio | Calificación |
|---|---:|
| Ciudad futurista visible | |
| Integración de naturaleza | |
| Presencia de personas | |
| Composición clara | |
| Iluminación cinematográfica | |
| Ausencia de texto | |
| Calidad general | |

### Diagnóstico
- ¿Qué funcionó?
- ¿Qué no funcionó?
- ¿Qué variable modificarás?
- ¿Por qué?


# 9. Prueba 2: modifica una variable

Cambia solamente **una** variable relevante.

Ejemplos:

- semilla;
- Guidance Scale;
- scheduler;
- número de pasos;
- una parte del prompt;
- negative prompt.


In [ ]:
# Copia la configuración de la prueba anterior
PROMPT_2 = PROMPT
NEGATIVE_PROMPT_2 = NEGATIVE_PROMPT
SEED_2 = 700                    # Ejemplo: cambia la semilla
STEPS_2 = STEPS
GUIDANCE_2 = GUIDANCE
SCHEDULER_2 = SCHEDULER_NAME

imagen_2, registro_2 = generar_prueba(
    numero_prueba=2,
    prompt=PROMPT_2,
    negative_prompt=NEGATIVE_PROMPT_2,
    seed=SEED_2,
    steps=STEPS_2,
    guidance=GUIDANCE_2,
    scheduler_name=SCHEDULER_2
)

display(imagen_2)
registro_2


## Evaluación de la prueba 2

- Variable modificada:
- Motivo:
- Cambio observado:
- ¿Mejoró o empeoró?
- Evidencia visual:


# 10. Prueba 3: segunda iteración

Modifica una variable con base en la evidencia de las pruebas anteriores.


In [ ]:
PROMPT_3 = PROMPT_2
NEGATIVE_PROMPT_3 = NEGATIVE_PROMPT_2
SEED_3 = SEED_2
STEPS_3 = 30                   # Ejemplo: cambia los pasos
GUIDANCE_3 = GUIDANCE_2
SCHEDULER_3 = SCHEDULER_2

imagen_3, registro_3 = generar_prueba(
    numero_prueba=3,
    prompt=PROMPT_3,
    negative_prompt=NEGATIVE_PROMPT_3,
    seed=SEED_3,
    steps=STEPS_3,
    guidance=GUIDANCE_3,
    scheduler_name=SCHEDULER_3
)

display(imagen_3)
registro_3


## Evaluación de la prueba 3

- Variable modificada:
- Motivo:
- Cambio observado:
- ¿La mejora justificó el costo?
- ¿Qué conservarías?


# 11. Prueba 4: propuesta final

Realiza una última modificación y genera tu propuesta definitiva.

La configuración debe respetar todas las restricciones.


In [ ]:
PROMPT_FINAL = PROMPT_3
NEGATIVE_PROMPT_FINAL = NEGATIVE_PROMPT_3
SEED_FINAL = SEED_3
STEPS_FINAL = STEPS_3
GUIDANCE_FINAL = 4.5           # Ejemplo de última modificación
SCHEDULER_FINAL = SCHEDULER_3

# Validación final
if not 4 <= GUIDANCE_FINAL <= 12:
    raise ValueError("Guidance Scale final fuera del rango permitido.")

if not 1 <= STEPS_FINAL <= 35:
    raise ValueError("Los pasos finales superan el máximo permitido.")

conceptos_finales = [
    x.strip() for x in NEGATIVE_PROMPT_FINAL.split(",") if x.strip()
]

if len(conceptos_finales) > 10:
    raise ValueError("El negative prompt final supera los 10 conceptos.")

imagen_final, registro_final = generar_prueba(
    numero_prueba=4,
    prompt=PROMPT_FINAL,
    negative_prompt=NEGATIVE_PROMPT_FINAL,
    seed=SEED_FINAL,
    steps=STEPS_FINAL,
    guidance=GUIDANCE_FINAL,
    scheduler_name=SCHEDULER_FINAL
)

display(imagen_final)
registro_final


# 12. Comparación de las cuatro propuestas


In [ ]:
imagenes_pruebas = [imagen_1, imagen_2, imagen_3, imagen_final]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for indice, (ax, imagen) in enumerate(zip(axes, imagenes_pruebas), start=1):
    ax.imshow(imagen)
    ax.set_title(f"Prueba {indice}")
    ax.axis("off")

plt.tight_layout()

ruta_comparacion = CARPETA_RESULTADOS / "comparacion_cuatro_pruebas.png"
plt.savefig(ruta_comparacion, dpi=150, bbox_inches="tight")
plt.show()

print("Comparación guardada en:", ruta_comparacion)


# 13. Historial experimental


In [ ]:
tabla_historial = pd.DataFrame(historial)[[
    "prueba",
    "seed",
    "steps",
    "guidance",
    "scheduler",
    "tiempo_segundos"
]]

tabla_historial


In [ ]:
ruta_csv = CARPETA_RESULTADOS / "historial_experimentos.csv"
pd.DataFrame(historial).to_csv(ruta_csv, index=False)

ruta_json = CARPETA_RESULTADOS / "configuracion_final.json"

with open(ruta_json, "w", encoding="utf-8") as archivo:
    json.dump(registro_final, archivo, ensure_ascii=False, indent=4)

print("Historial guardado en:", ruta_csv)
print("Configuración final guardada en:", ruta_json)


# 14. Selección y justificación final

Completa:

### Imagen seleccionada
Prueba número:

### Prompt final
Copia el prompt utilizado.

### Negative prompt final
Copia las restricciones utilizadas.

### Semilla
¿Por qué es importante conservarla?

### Guidance Scale
¿Por qué elegiste ese valor?

### Pasos
¿La calidad justificó el tiempo?

### Scheduler
¿Por qué fue adecuado para el objetivo?

### Evaluación final
¿Qué requisitos cumplió y cuáles no?

### Conclusión técnica
Explica en máximo 150 palabras cómo construiste el resultado.


# 15. Prueba de reproducibilidad

Generaremos nuevamente la configuración final.

Si la configuración está correctamente registrada, el resultado debe reproducirse.


In [ ]:
pipe.scheduler = SCHEDULERS[SCHEDULER_FINAL]

generator = torch.Generator(device="cuda").manual_seed(SEED_FINAL)

imagen_reproducida = pipe(
    prompt=PROMPT_FINAL,
    negative_prompt=NEGATIVE_PROMPT_FINAL,
    num_inference_steps=STEPS_FINAL,
    guidance_scale=GUIDANCE_FINAL,
    width=WIDTH,
    height=HEIGHT,
    generator=generator
).images[0]

ruta_reproduccion = CARPETA_RESULTADOS / "reproduccion_final.png"
imagen_reproducida.save(ruta_reproduccion)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(imagen_final)
axes[0].set_title("Propuesta final")
axes[0].axis("off")

axes[1].imshow(imagen_reproducida)
axes[1].set_title("Reproducción")
axes[1].axis("off")

plt.tight_layout()
plt.show()


# 16. Verificación automática de reproducción


In [ ]:
import numpy as np

array_final = np.array(imagen_final).astype(np.int16)
array_reproducida = np.array(imagen_reproducida).astype(np.int16)

diferencia = np.abs(array_final - array_reproducida)

print("Diferencia máxima:", diferencia.max())
print("Diferencia promedio:", diferencia.mean())

if diferencia.max() == 0:
    print("La propuesta final fue reproducida exactamente.")
else:
    print(
        "Se detectaron diferencias. Revisa que todos los parámetros "
        "y el entorno sean iguales."
    )


# 17. Preguntas integradoras

1. ¿Qué parámetro produjo el cambio más evidente?
2. ¿Qué parámetro fue más difícil de interpretar?
3. ¿La mejor imagen apareció en la primera prueba?
4. ¿Qué ventaja ofreció conservar la semilla?
5. ¿Por qué no conviene modificar todos los parámetros simultáneamente?
6. ¿Qué papel tuvo el negative prompt?
7. ¿Qué decisión equilibró mejor calidad y tiempo?
8. ¿Qué información debe documentarse para reproducir una imagen?
9. ¿Puede existir una única configuración óptima para todos los prompts?
10. ¿En qué sentido esta actividad se parece a un experimento científico?


## Reflexión
> “La competencia no consiste en memorizar valores, sino en formular hipótesis, probarlas, observar evidencia y justificar decisiones.”


# 18. Evidencia de aprendizaje

Entrega:

1. comparación de las cuatro pruebas;
2. historial de parámetros;
3. imagen final;
4. configuración final en JSON;
5. prueba de reproducibilidad;
6. respuestas integradoras;
7. conclusión técnica.



# 19. Cierre conceptual

Una generación reproducible requiere documentar:

- modelo;
- prompt;
- negative prompt;
- semilla;
- Guidance Scale;
- pasos;
- scheduler;
- tamaño de imagen.

La idea final de la sesión es:

> **Una buena generación no es solamente atractiva: también es intencional, justificable y reproducible.**
